# Tavily Raw vs Cleaned Content Test

In [8]:
import os
import re
from tavily import TavilyClient
os.environ["TAVILY_API_KEY"] = "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"

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Two test queries that will hit content-rich pages
queries = [
    "GitHub Copilot pricing plans features 2025",
    "AI code review tools market size 2025",
]

raw_results = {}
for q in queries:
    print(f"Searching: {q}")
    resp = client.search(q, search_depth="advanced", max_results=1, include_raw_content=True)
    r = resp["results"][0]
    raw_results[q] = r
    print(f"  -> {r['title']}")
    print(f"  -> {r['url']}")
    print(f"  -> Summary: {len(r['content'])} chars")
    print(f"  -> Raw:     {len(r.get('raw_content') or '')} chars")
    print()

Searching: GitHub Copilot pricing plans features 2025  -> Understanding GitHub Copilot pricing: A complete 2025 guide
  -> https://www.eesel.ai/blog/github-copilot-pricing
  -> Summary: 1978 chars
  -> Raw:     18102 chars

Searching: AI code review tools market size 2025
  -> AI Code Review Tool Market's Decade Ahead 2025-2033
  -> https://www.marketreportanalytics.com/reports/ai-code-review-tool-73093
  -> Summary: 799 chars
  -> Raw:     43349 chars


## Cell 1: Tavily RAW Content (unprocessed)

In [9]:
# FULL RAW CONTENT — everything Tavily returns, unprocessed
for q, r in raw_results.items():
    raw = r.get("raw_content") or "(no raw content returned)"
    print("=" * 80)
    print(f"QUERY: {q}")
    print(f"TITLE: {r['title']}")
    print(f"URL:   {r['url']}")
    print(f"RAW CONTENT LENGTH: {len(raw)} chars")
    print("=" * 80)
    print()
    print(raw)
    print()
    print()


QUERY: GitHub Copilot pricing plans features 2025
TITLE: Understanding GitHub Copilot pricing: A complete 2025 guide
URL:   https://www.eesel.ai/blog/github-copilot-pricing
RAW CONTENT LENGTH: 18102 chars

* [Product](# "Product")
  + **AI agent**

    Automate frontline support
  + **AI copilot**

    Draft replies and assistance
  + **AI triage**

    Route, edit or tag tickets
  + **AI chatbot**

    Chatbot on your site
  + **AI internal chat**

    Instant answers for your team
  + **AI email assistant**

    Instant email & ticket drafts
  + **AI blog writer**

    Generate high quality blogs
  + **AI chatbot e-commerce**

    Answers and guides shoppers
* [Integrations](# "Integrations")



  + **Google Docs**
  + **Shopify**
  + **Explore all integrations**

    Over 100+ apps supported
* [Solutions](# "Solutions")
  + **AI for Chatbot Ecommerce**

    AI live chat for ecommerce
  + **AI for Agent Assist**

    Assist your agents in real time
  + **AI for Customer Support Autom

## Cell 2: Tavily CLEANED Content (after clean_raw_content)

In [10]:
def clean_raw_content(text: str) -> str:
    """Strip web artifacts from Tavily raw_content, keep useful text."""
    if not text:
        return ""

    # Cut everything after common report boilerplate markers
    cutoff_patterns = [
        r'^## List of (?:Figures|Tables)',
        r'^## Frequently Asked Questions',
        r'^## Methodology',
        r'^Related Reports',
        r'^## List of Figures',
    ]
    for pattern in cutoff_patterns:
        match = re.search(pattern, text, flags=re.MULTILINE)
        if match:
            text = text[:match.start()]

    # Remove markdown image tags: ![alt](url)
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)

    # Remove markdown links but keep the text: [text](url) -> text
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)

    # Remove raw URLs on their own line
    text = re.sub(r'^https?://\S+$', '', text, flags=re.MULTILINE)

    # Remove navigation-style lines (short lines with just links/bullets)
    text = re.sub(r'^\s*[\*\+\-]\s+\w[\w\s]{0,35}$', '', text, flags=re.MULTILINE)

    # Remove common boilerplate phrases
    boilerplate = [
        r'Sign in.*?$',
        r'Sign up.*?$',
        r'Subscribe.*?newsletter.*?$',
        r'Cookie.*?policy.*?$',
        r'Privacy.*?policy.*?$',
        r'Terms.*?service.*?$',
        r'All rights reserved.*?$',
        r'Share this.*?$',
        r'Table of [Cc]ontents?',
        r'Download [Ff]ree [Ss]ample',
        r'Request [Ss]ample',
        r'Buy [Nn]ow',
        r'Add to [Cc]art',
        r'Contact [Uu]s',
        r'Learn [Mm]ore\s*[>]?\s*$',
        r'Read [Mm]ore\s*[>]?\s*$',
    ]
    for pattern in boilerplate:
        text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)

    # Remove HTML entities that survived
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#x?[0-9a-fA-F]+;', ' ', text)

    # Collapse multiple blank lines into one
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)

    # Strip leading/trailing whitespace per line
    text = '\n'.join(line.strip() for line in text.split('\n'))

    # Remove lines that are just punctuation or whitespace
    text = '\n'.join(
        line for line in text.split('\n')
        if line.strip() and not re.match(r'^[\s\*\-\+\#\>\|\=\~]+$', line.strip())
    )

    return text.strip()


# FULL CLEANED CONTENT — after removing nav, boilerplate, image tags, etc.
for q, r in raw_results.items():
    raw = r.get("raw_content") or ""
    cleaned = clean_raw_content(raw)
    reduction = (1 - len(cleaned) / len(raw)) * 100 if raw else 0

    print("=" * 80)
    print(f"QUERY: {q}")
    print(f"TITLE: {r['title']}")
    print(f"URL:   {r['url']}")
    print(f"RAW: {len(raw):,} chars -> CLEANED: {len(cleaned):,} chars ({reduction:.0f}% removed)")
    print("=" * 80)
    print()
    print(cleaned)
    print()
    print()

QUERY: GitHub Copilot pricing plans features 2025
TITLE: Understanding GitHub Copilot pricing: A complete 2025 guide
URL:   https://www.eesel.ai/blog/github-copilot-pricing
RAW: 18,102 chars -> CLEANED: 13,403 chars (26% removed)

+ **AI agent**
Automate frontline support
+ **AI copilot**
Draft replies and assistance
+ **AI triage**
Route, edit or tag tickets
+ **AI chatbot**
Chatbot on your site
+ **AI internal chat**
Instant answers for your team
+ **AI email assistant**
Instant email & ticket drafts
+ **AI blog writer**
Generate high quality blogs
+ **AI chatbot e-commerce**
Answers and guides shoppers
+ **Google Docs**
+ **Shopify**
+ **Explore all integrations**
Over 100+ apps supported
+ **AI for Chatbot Ecommerce**
AI live chat for ecommerce
+ **AI for Agent Assist**
Assist your agents in real time
+ **AI for Customer Support Automation**
Answer customer questions
+ **AI for Service Desk**
Answer service desk queries
+ **AI for IT Operations**
Help your team operates
+ **AI for 